# Geometry explorer: disc and bicone relative to the line of sight

Draws a schematic of any disc + hollow-bicone geometry, in the spirit of Fischer et al. (2013, ApJS 209, 1): useful
to build intuition, to check that a fitted geometry is physically sensible, or to make a figure.

**Two views.** *View from the observer* (sky plane, north up, east left): parts of the cones **behind** the disc are drawn
as a wire-frame, parts **in front** are shaded, and yellow lines mark where the cones cross the disc plane. *3-D view*:
an oblique view with the direction to the observer marked.

**Conventions** (the same as the kinematic models of the package):

| Parameter | Meaning |
|---|---|
| `pa_out` | position angle (east of north) of the **redshifted** (receding) cone |
| `inc_out` | angle between the bicone axis and the **line of sight** (0 = pole-on, 90 = in the sky plane) |
| `theta_in`, `theta_out` | inner / outer opening angles of the hollow bicone, from the axis. By default the outer angle is **drawn as `theta_out / 2`** (the model applies these angles to sky-projected positions, so full-size 3-D cones look too wide); `HALVE_THETA_OUT = False` draws it unchanged |
| `pa_disc` | position angle of the **receding** side of the disc major axis |
| `inc_disc` | disc inclination (0 = face-on, 90 = edge-on) |
| `R_out` | the cones are drawn out to 2 `R_out` (where the model outflow velocity drops to 0); or set `LENGTH_PC` |
| `DISC_NEAR_SIDE` | +1: the side of the disc at `pa_disc` + 90 deg is nearer to us; -1: `pa_disc` - 90 deg |

Fischer et al. measure the bicone inclination from the **plane of the sky**: `inc_out = 90 - i_Fischer`, and their
"inclined toward Earth" end is the **blueshifted** cone, so `pa_out` points to the other end. Their opening angles are
already 3-D half-opening angles, so draw them with `halve_theta_out=False`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
try:
    import breaking_the_blend as btb
except ModuleNotFoundError:   # not installed (pip install -e .): use the copy in this repository
    import sys, os
    sys.path.insert(0, os.path.abspath(".."))
    import breaking_the_blend as btb
from breaking_the_blend import plotting as bplt     # also sets the publication plotting style

SAVE_FIGURES = "none"            # "png", "pdf", "both" or "none"
proj = btb.Project("geometry_explorer", results_dir="results", save_figures=SAVE_FIGURES)

## 1. Define a geometry - EDIT

In [ ]:
GEOMETRY = dict(
    inc_disc=45.0,   pa_disc=30.0,          # disc: inclination, PA of the receding side [deg]
    inc_out=60.0,    pa_out=120.0,          # bicone: inclination from the line of sight, PA of the redshifted cone [deg]
    theta_in=15.0,   theta_out=40.0,        # inner / outer half-opening angles [deg]
    R_out=500.0,                            # [pc] cones drawn out to 2 R_out
)
DISC_NEAR_SIDE = +1
HALVE_THETA_OUT = True    # draw the outer opening angle as theta_out / 2 (see the table above)
LENGTH_PC = None          # override the drawn cone length [pc]
DISC_RADIUS_PC = None     # override the drawn disc radius [pc] (default 1.25 x cone length)
ELEV, AZIM = 35, -75      # viewing angles of the 3-D panel

fig = btb.plot_geometry(GEOMETRY, length_pc=LENGTH_PC, disc_radius_pc=DISC_RADIUS_PC,
                        disc_near_side=DISC_NEAR_SIDE, elev=ELEV, azim=AZIM, title="My geometry",
                        halve_theta_out=HALVE_THETA_OUT)
proj.savefig(fig, "my_geometry")

## 2. How the view changes with the bicone inclination

The same bicone seen at different inclinations to the line of sight (compare Figs. 7-8 of Fischer et al. 2013). Only
the view from the observer is shown; change `INCLINATIONS` or any other entry.

In [ ]:
INCLINATIONS = [10, 40, 70, 90]
fig, axes = plt.subplots(1, len(INCLINATIONS), figsize=(4.6 * len(INCLINATIONS), 4.8), constrained_layout=True)
for ax, inc in zip(axes, INCLINATIONS):
    btb.plot_sky_view({**GEOMETRY, "inc_out": inc}, ax=ax, disc_near_side=DISC_NEAR_SIDE, legend=(inc == INCLINATIONS[0]),
                      title=f"inc_out = {inc} deg", halve_theta_out=HALVE_THETA_OUT)
proj.savefig(fig, "inclination_sequence")

## 3. Disc only, or bicone only

`show_disc=False` draws the bicone alone (no occlusion); a bicone with `theta_out` close to 0 effectively hides it.

In [ ]:
fig = btb.plot_geometry(GEOMETRY, disc_near_side=DISC_NEAR_SIDE, show_disc=False, title="Bicone only", print_params=False,
                        halve_theta_out=HALVE_THETA_OUT)

## 4. Examples

* **NGC 5728** - the geometry adopted in Veenema et al. (2026) (disc from Shimizu et al. 2019), drawn with
  `theta_out / 2` as in notebooks 01 and 02.
* **Circinus** - from Table 6 of Fischer et al. (2013), converted to the conventions above (`inc_out = 90 - 25 = 65`,
  `pa_out` pointing to the end *not* inclined toward Earth). Its opening angles are already half-opening angles, so it
  is drawn with `halve_theta_out=False`. The near side of the host disc is not given there - flip `disc_near_side`
  to compare. This illustrates the conventions; it is not a re-derived fit.

In [ ]:
EXAMPLES = {   # name: (geometry, halve_theta_out)
    "NGC 5728 (Veenema et al. 2026)": (dict(inc_disc=43, pa_disc=194, inc_out=48, pa_out=110, theta_in=21, theta_out=80,
                                            R_out=600), True),
    "Circinus (Fischer et al. 2013)": (dict(inc_disc=65, pa_disc=30, inc_out=65, pa_out=128, theta_in=36, theta_out=41,
                                            R_out=20), False),
}
for name, (geo, halve) in EXAMPLES.items():
    fig = btb.plot_geometry(geo, disc_near_side=+1, title=name, halve_theta_out=halve)
    proj.savefig(fig, "example_" + name.split(" (")[0].replace(" ", "_"))

## 5. Plotting the geometry of a fit or a decomposition

In notebooks 01 and 02 the same function is called with fitted or adopted parameters:

```python
fig = btb.plot_geometry(btb.mean_geometry(combined_fits), disc_near_side=+1)    # best-fit geometry
fig = btb.plot_geometry(btb.mean_geometry(line_parameters))                    # geometry used in the decomposition
fig = btb.plot_geometry(combined_fits["NeIII"].params)                         # a single line's fit
```

The individual panels are also available as `btb.plot_sky_view(params, ax=...)` and
`btb.plot_3d_view(params, ax=...)` (pass a 3-D axis) for building custom figures.